# Transformer ML Stack (Entry->Exit Seq2Seq)

Standalone notebook for the transformer workflow: universe/data build, labels, entry->exit text dataset construction, and Seq2Seq training.


In [1]:
# ============================================================
# 0) Environment + Universe + Context
# ============================================================
import os
from dotenv import load_dotenv

import numpy as np
import pandas as pd

from modules.data.universe_fmp import build_large_liquid_universe_single_call
from modules.data.context import DataContext

load_dotenv()
FMP_API_KEY = os.getenv("FMP_API_KEY")
if not FMP_API_KEY:
    raise RuntimeError("Missing FMP_API_KEY in environment/.env")

# Screener universe, then keep first 7 for fast iteration
universe_full = build_large_liquid_universe_single_call(
    api_key=FMP_API_KEY,
    marketCapMoreThan=10_000_000_000,
    country="US",
    exchange="NASDAQ,NYSE,AMEX",
    priceMoreThan=5,
)

universe = tuple(universe_full)
universe = tuple(universe_full[:7])

DATA_DIR = "./data"
ctx = DataContext.from_data_dir(
    api_key=FMP_API_KEY,
    data_dir=DATA_DIR,
    db_name="quant.db",
    sleep_s=0.0,
    history_years=100,
)

print("Universe size:", len(universe))
print("Universe:", universe)


Universe size: 7
Universe: ('A', 'AA', 'AAFTX', 'AAGTX', 'AAHTX', 'AALTX', 'AANTX')


In [2]:
# ============================================================
# 1) Build Technical / Fundamental / Macro Layers
# ============================================================
import pandas as pd
from modules.api import (
    build_technical_dataframe,
    build_fundamental_dataframe,
    build_macro_dataframe,
    _summarize_dataset_for_llm,
    summarize_technical_features,
)
from modules.features.macro_fmp import MacroFeatureConfig

start_date = "1980-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

# -----------------------------
# Technical
# -----------------------------
technical_df, technical_cols = build_technical_dataframe(
    ctx=ctx,
    symbols=list(universe),
    start_date=start_date,
    end_date=end_date,
    verbose_data=True,
)

# Dedicated technical-only diagnostics
summarize_technical_features(technical_df, technical_cols)

# -----------------------------
# Fundamentals
# -----------------------------
fund_df, fund_cols = build_fundamental_dataframe(
    ctx=ctx,
    symbols=list(universe),
    start_date=start_date,
    end_date=end_date,
    period="quarter",
    target_index=technical_df.index,
    daily_prices=technical_df,
    verbose=True,
)

# -----------------------------
# Macro
# -----------------------------
macro_cfg = MacroFeatureConfig(
    economic_series=("CPI", "UNEMPLOYMENT", "GDP", "INFLATION", "FEDERAL_FUNDS_RATE"),
    include_treasury_rates=True,
    fill_method="ffill",
)
macro_df, macro_cols = build_macro_dataframe(
    ctx=ctx,
    start_date=start_date,
    end_date=end_date,
    config=macro_cfg,
    target_index=technical_df.index,
    verbose=True,
)

# -----------------------------
# Merge layers
# -----------------------------
final_df = pd.concat([technical_df, fund_df, macro_df], axis=1)
all_features = sorted(set(technical_cols + fund_cols + macro_cols))

# Drop all-NaN numeric features
all_nan_numeric = [
    c for c in all_features
    if c in final_df.columns
    and pd.api.types.is_numeric_dtype(final_df[c])
    and final_df[c].isna().all()
]
if all_nan_numeric:
    final_df = final_df.drop(columns=all_nan_numeric)
    all_features = [c for c in all_features if c not in all_nan_numeric]

# Final combined diagnostics
_summarize_dataset_for_llm(final_df, all_features)
print("Final shape:", final_df.shape)
print("Total active features:", len(all_features))


[A] skip FMP check (sqlite max 2026-02-18 >= latest_possible 2026-02-18)
[A] coverage: rows=6,601 start=1999-11-18 end=2026-02-18
[AA] skip FMP check (sqlite max 2026-02-18 >= latest_possible 2026-02-18)
[AA] coverage: rows=7,552 start=1996-02-13 end=2026-02-18
[AAFTX] skip FMP check (sqlite max 2026-02-18 >= latest_possible 2026-02-18)
[AAFTX] coverage: rows=4,791 start=2007-02-02 end=2026-02-18
[AAGTX] skip FMP check (sqlite max 2026-02-18 >= latest_possible 2026-02-18)
[AAGTX] coverage: rows=4,791 start=2007-02-02 end=2026-02-18
[AAHTX] skip FMP check (sqlite max 2026-02-18 >= latest_possible 2026-02-18)
[AAHTX] coverage: rows=4,791 start=2007-02-02 end=2026-02-18
[AALTX] skip FMP check (sqlite max 2026-02-18 >= latest_possible 2026-02-18)
[AALTX] coverage: rows=4,791 start=2007-02-02 end=2026-02-18
[AANTX] skip FMP check (sqlite max 2026-02-18 >= latest_possible 2026-02-18)
[AANTX] coverage: rows=2,740 start=2015-03-27 end=2026-02-18

  LLM DATA DEBUG REPORT  
SHAPE:       33,859 r

In [3]:
# ============================================================
# 2) Build Label Frame
# ============================================================
from modules.api import build_label_dataframe

K_PARAMS = {"YE": [1, 2, 4, 8]}
EXECUTION_PARAMS = {"price_col": "close", "fee_bps": 5.0, "slippage_bps": 5.0}
WEIGHTING_PARAMS = {
    "use_sample_weight": True,
    "alpha": 4.0,
    "r_clip": 0.10,
    "horizon_balance": True,
}

symbols_in_df = set(technical_df.index.get_level_values("symbol"))
daily_map = {s: technical_df.xs(s, level="symbol") for s in universe if s in symbols_in_df}

label_df = build_label_dataframe(
    daily_by_symbol=daily_map,
    k_params=K_PARAMS,
    execution_params=EXECUTION_PARAMS,
    weighting=WEIGHTING_PARAMS,
    add_rank_labels=True,
    verbose=True,
)

print("Label rows:", len(label_df))
print("Label cols:", list(label_df.columns))


  ORACLE LABEL PERFORMANCE & DEDUPLICATION SUMMARY
DEDUPLICATION METRICS:
  - Raw Signal Count:    8,376
  - Unique Signal Count: 4,425
  - Redundancy Removed:  47.2%
--------------------------------------------------------------------------------
Horizon  Side  Trades  Mean Return %  Win Rate %  Avg Duration
  YE_k1 SHORT      80          23.74       100.0         135.7
  YE_k1   BUY      99          42.94       100.0         251.2
  YE_k2 SHORT     109          15.26       100.0          62.8
  YE_k2   BUY     111          25.34       100.0         119.0
  YE_k4 SHORT     231          10.87       100.0          31.2
  YE_k4   BUY     235          15.49       100.0          53.6
  YE_k8 SHORT     678           6.78       100.0          13.8
  YE_k8   BUY     668           8.44       100.0          21.8

Label rows: 4425
Label cols: ['target', 'side', 'horizon', 'trade_return', 'sample_weight', 'label', 'market_position', 'trade_duration_days', 'rank_y', 'side_profit']


In [4]:
# ============================================================
# 2.5) Build Entry->Exit Seq2Seq Dataset
# ============================================================
import pandas as pd
from modules.data.preparation import Entry2ExitTextConfig, prepare_entry2exit_dataset
from modules.labels.trades import labels_panel_to_trades_df
from transformers import AutoTokenizer
from flair.data import Sentence
from flair.tokenization import SegtokTokenizer

# 1) Build trades and enrich with entry/exit prices from final_df
trades_df = labels_panel_to_trades_df(label_df).copy()

# final_df columns are PascalCase; resolve requested OHLC column case-insensitively
requested_price_col = "open"  # change to "close" if preferred
price_col = next((c for c in final_df.columns if str(c).lower() == requested_price_col.lower()), None)
if price_col is None:
    available = [c for c in final_df.columns if str(c).lower() in {"open", "high", "low", "close"}]
    raise KeyError(f"Price column '{requested_price_col}' not found. Available OHLC columns: {available}")

px = final_df[[price_col]].reset_index().rename(columns={price_col: "px"})
px["date"] = pd.to_datetime(px["date"], errors="coerce").dt.normalize()
px["symbol"] = px["symbol"].astype(str)

trades_df["entry_date"] = pd.to_datetime(trades_df["entry_date"], errors="coerce").dt.normalize()
trades_df["exit_date"] = pd.to_datetime(trades_df["exit_date"], errors="coerce").dt.normalize()
trades_df["symbol"] = trades_df["symbol"].astype(str)

entry_px_df = px.rename(columns={"date": "entry_date", "px": "entry_px"})
exit_px_df = px.rename(columns={"date": "exit_date", "px": "exit_px"})

trades_df = trades_df.merge(
    entry_px_df[["symbol", "entry_date", "entry_px"]],
    on=["symbol", "entry_date"],
    how="left",
)
trades_df = trades_df.merge(
    exit_px_df[["symbol", "exit_date", "exit_px"]],
    on=["symbol", "exit_date"],
    how="left",
)

# 2) Build entry->exit text dataset
entry2exit_df = prepare_entry2exit_dataset(
    features_df=final_df,
    trades_df=trades_df,
    config=Entry2ExitTextConfig(
        numeric_precision=2,
        scientific_for_large_numbers=True,
        scientific_threshold=1_000_000.0,
        dedupe_source_duplicate_features=True,
        drop_missing_entry_rows=True,
        target_fields=("entry_px", "exit_px"),  # removed trade_return/trade_duration_days
    ),
)


# 3) Token counts with both model tokenizer and Flair tokenizer
hf_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
flair_tokenizer = SegtokTokenizer()

def flair_token_count(text: str) -> int:
    sent = Sentence(str(text), use_tokenizer=flair_tokenizer)
    return len(sent.tokens)

entry2exit_df["input_token_count_model"] = entry2exit_df["input_text"].apply(
    lambda s: len(hf_tokenizer(str(s), add_special_tokens=True)["input_ids"])
)
entry2exit_df["target_token_count_model"] = entry2exit_df["target_text"].apply(
    lambda s: len(hf_tokenizer(str(s), add_special_tokens=True)["input_ids"])
)

entry2exit_df["input_token_count_flair"] = entry2exit_df["input_text"].apply(flair_token_count)
entry2exit_df["target_token_count_flair"] = entry2exit_df["target_text"].apply(flair_token_count)

# 4) Quick checks
display(entry2exit_df.head(5))
print("Entry2Exit rows:", len(entry2exit_df))
print("\nExample input_text:\n", entry2exit_df.loc[0, "input_text"][:500], "...")
print("\nExample target_text:\n", entry2exit_df.loc[0, "target_text"])

# Optional: save
# entry2exit_df.to_csv("entry2exit.csv", index=False)

# Numeric summary
entry2exit_df.describe(include="all")


Token indices sequence length is longer than the specified maximum sequence length for this model (1644 > 512). Running this sequence through the model will result in indexing errors


,symbol,entry_date,exit_date,side,horizon,trade_return,trade_duration_days,sample_weight,entry_px,exit_px,input_text,target_text,input_token_count_model,target_token_count_model,input_token_count_flair,target_token_count_flair
0,A,2001-02-16,2001-09-27,short,YE_k1,0.613467,223,10.284254,28.06,10.69,Symbol=A EntryDate=2001-02-16 Open=28.06 High=...,Symbol=A ExitDate=2001-09-27 Open=10.69 High=1...,1644,1717,613,625
1,A,2001-02-26,2001-03-07,short,YE_k8,0.226795,14,1.000000,21.15,23.74,Symbol=A EntryDate=2001-02-26 Open=21.15 High=...,Symbol=A ExitDate=2001-03-07 Open=23.74 High=2...,1656,1651,613,619
2,A,2001-02-26,2001-03-07,long,YE_k8,0.067800,9,3.711996,21.15,23.74,Symbol=A EntryDate=2001-02-26 Open=21.15 High=...,Symbol=A ExitDate=2001-03-07 Open=23.74 High=2...,1656,1651,613,619
3,A,2001-03-21,2001-03-26,short,YE_k8,0.248000,11,1.000000,18.53,20.82,Symbol=A EntryDate=2001-03-21 Open=18.53 High=...,Symbol=A ExitDate=2001-03-26 Open=20.82 High=2...,1680,1674,613,619
4,A,2001-03-21,2001-03-26,long,YE_k8,0.171153,5,5.000000,18.53,20.82,Symbol=A EntryDate=2001-03-21 Open=18.53 High=...,Symbol=A ExitDate=2001-03-26 Open=20.82 High=2...,1680,1674,613,619


Entry2Exit rows: 2197

Example input_text:
 Symbol=A EntryDate=2001-02-16 Open=28.06 High=28.33 Low=27.3 Close=28.06 Volume=5.29e+06 Ret1d=-0.07 Ret2d=0.04 Ret3d=0.06 Ret5d=-0.05 Ret10d=-0.04 Ret20d=-0.24 Ret63d=0.15 Ret126d=0.08 Ret252d=-0.46 CumRet5d=-0.05 CumRet10d=-0.04 CumRet20d=-0.24 CumRet63d=0.15 DistSMA5=0 SMASlope5=-0.28 DistSMA10=-0.01 SMASlope10=-0.12 DistSMA20=-0.07 SMASlope20=-0.44 DistSMA50=-0.1 SMASlope50=-0.06 DistSMA100=-0.03 SMASlope100=0.01 DistSMA200=-0.13 SMASlope200=-0.12 DistEMA12=-0.03 DistEMA26=-0.06 DistEMA50=-0 ...

Example target_text:
 Symbol=A ExitDate=2001-09-27 Open=10.69 High=10.79 Low=10.33 Close=10.79 Volume=2.68e+06 Ret1d=0 Ret2d=-0.03 Ret3d=-0.08 Ret5d=-0.08 Ret10d=-0.18 Ret20d=-0.26 Ret63d=-0.35 Ret126d=-0.49 Ret252d=-0.62 CumRet5d=-0.08 CumRet10d=-0.18 CumRet20d=-0.26 CumRet63d=-0.35 DistSMA5=-0.03 SMASlope5=-0.18 DistSMA10=-0.08 SMASlope10=-0.23 DistSMA20=-0.18 SMASlope20=-0.19 DistSMA50=-0.28 SMASlope50=-0.13 DistSMA100=-0.37 SMASlope100=-0.11 

,symbol,entry_date,exit_date,side,horizon,trade_return,trade_duration_days,sample_weight,entry_px,exit_px,input_text,target_text,input_token_count_model,target_token_count_model,input_token_count_flair,target_token_count_flair
count,2197,2197,2197,2197,2197,2197.000000,2197.000000,2197.000000,2197.000000,2197.000000,2197,2197,2197.000000,2197.000000,2197.000000,2197.000000
unique,7,NaN,NaN,2,4,NaN,NaN,NaN,NaN,NaN,1765,2004,NaN,NaN,NaN,NaN
top,AA,NaN,NaN,long,YE_k8,NaN,NaN,NaN,NaN,NaN,Symbol=AA EntryDate=1998-10-05 Open=26.2 High=...,Symbol=AAHTX ExitDate=2025-10-10 Open=22.86 Hi...,NaN,NaN,NaN,NaN
freq,470,NaN,NaN,1107,1343,NaN,NaN,NaN,NaN,NaN,2,2,NaN,NaN,NaN,NaN
mean,NaN,2015-07-10 18:33:35.475648512,2015-10-07 01:14:43.204369664,NaN,NaN,0.121389,44.944470,2.564302,23.482635,23.991370,NaN,NaN,894.381884,911.254438,379.682749,386.185253
min,NaN,1997-05-22 00:00:00,1997-06-24 00:00:00,NaN,NaN,0.008097,1.000000,1.000000,2.900000,2.900000,NaN,NaN,571.000000,586.000000,289.000000,295.000000
25%,NaN,2010-06-21 00:00:00,2010-11-04 00:00:00,NaN,NaN,0.038956,8.000000,1.000000,8.240000,8.380000,NaN,NaN,608.000000,624.000000,294.000000,300.000000
50%,NaN,2016-05-19 00:00:00,2016-07-18 00:00:00,NaN,NaN,0.068485,19.000000,1.610621,14.880000,14.990000,NaN,NaN,638.000000,653.000000,294.000000,300.000000
75%,NaN,2021-03-17 00:00:00,2021-07-19 00:00:00,NaN,NaN,0.141583,44.000000,3.858776,26.030000,25.990000,NaN,NaN,1631.000000,1651.000000,619.000000,625.000000
max,NaN,2026-02-09 00:00:00,2026-02-12 00:00:00,NaN,NaN,3.121616,360.000000,10.845336,172.260000,172.260000,NaN,NaN,1723.000000,1739.000000,625.000000,631.000000


In [ ]:
# ============================================================
# Train Encoder-Decoder (Seq2Seq) on Entry->Exit Dataset
# + Subtoken Pooling Inference (Flair-style word alignment)
# ============================================================

# If needed once:
# !pip install -q transformers datasets accelerate sentencepiece flair

import os
import numpy as np
import pandas as pd
import torch

from flair.tokenization import SegtokTokenizer
from transformers import AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput

from modules.models.base import FitSpec, SequenceSpec
from modules.models.transformers.seq2seq import train_seq2seq_model

# ------------------------------------------------------------------
# 0) Use your existing entry2exit_df (must have input_text/target_text)
# ------------------------------------------------------------------
required_cols = {"input_text", "target_text"}
if "entry2exit_df" not in globals():
    raise RuntimeError("entry2exit_df is not defined. Build it first.")
if not required_cols.issubset(entry2exit_df.columns):
    raise RuntimeError(f"entry2exit_df must contain {required_cols}.")

seq2seq_df = entry2exit_df.copy()
seq2seq_df = seq2seq_df.dropna(subset=["input_text", "target_text"]).reset_index(drop=True)
seq2seq_df = seq2seq_df[
    seq2seq_df["input_text"].astype(str).str.len().gt(0)
    & seq2seq_df["target_text"].astype(str).str.len().gt(0)
].reset_index(drop=True)

print("Total rows:", len(seq2seq_df))
display(seq2seq_df[["input_text", "target_text"]].head(3))

# ------------------------------------------------------------------
# 1) Time-aware split (no random leakage)
# ------------------------------------------------------------------
if "entry_date" in seq2seq_df.columns:
    seq2seq_df["entry_date"] = pd.to_datetime(seq2seq_df["entry_date"], errors="coerce")
    seq2seq_df = seq2seq_df.sort_values("entry_date").reset_index(drop=True)

split_idx = int(len(seq2seq_df) * 0.90)
train_df = seq2seq_df.iloc[:split_idx].copy()
val_df = seq2seq_df.iloc[split_idx:].copy()

print(f"Train rows: {len(train_df)}")
print(f"Val rows:   {len(val_df)}")

# ------------------------------------------------------------------
# 2) Define sequence spec + train
# ------------------------------------------------------------------
seq_spec = SequenceSpec(
    input_col="input_text",
    target_col="target_text",
    padding="longest",
    max_source_length=128,
    max_target_length=64,
)

fit_spec = FitSpec(
    feature_cols=[],
    task_type="seq2seq",
    sequence=seq_spec,
    model_tag="entry2exit_flan_t5_small",
)

artifacts = train_seq2seq_model(
    seq2seq_df=train_df[["input_text", "target_text"]].copy(),
    model_name="google/flan-t5-small",
    max_length=128,
    num_train_epochs=10.0,
    batch_size=8,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="constant_with_warmup",
    spec=fit_spec,
)

model = artifacts["model"].eval()

# create tokenizer explicitly here (fast tokenizer required for word_ids)
hf_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small", use_fast=True)
tokenizer = hf_tokenizer

# ------------------------------------------------------------------
# 3) Save model + tokenizer
# ------------------------------------------------------------------
save_dir = "./artifacts/entry2exit_flan_t5_small"
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
hf_tokenizer.save_pretrained(save_dir)
print("Saved model to:", save_dir)

# ------------------------------------------------------------------
# 4) Subtoken pooling utilities + generation
# ------------------------------------------------------------------
flair_tok = SegtokTokenizer()

def _pool_subtokens_with_specials(hidden, word_ids, attention_mask, strategy="mean"):
    # hidden: [seq_len, hidden_dim]
    pooled = []
    used_word_ids = set()

    for i, wid in enumerate(word_ids):
        if int(attention_mask[i].item()) == 0:
            continue

        if wid is None:
            # Keep special token states (<pad>, </s>, etc.) as-is.
            pooled.append(hidden[i])
            continue

        if wid in used_word_ids:
            continue

        idxs = [
            j for j, w in enumerate(word_ids)
            if w == wid and int(attention_mask[j].item()) == 1
        ]
        if not idxs:
            continue

        sub = hidden[idxs]  # [n_subtokens, H]
        if strategy == "first":
            vec = sub[0]
        elif strategy == "last":
            vec = sub[-1]
        elif strategy == "mean":
            vec = sub.mean(dim=0)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        pooled.append(vec)
        used_word_ids.add(wid)

    if not pooled:
        return None
    return torch.stack(pooled, dim=0)  # [pooled_seq_len, H]


def generate_exit_text(
    input_text: str,
    max_new_tokens: int = 128,
    num_beams: int = 1,
    subtoken_pooling: str = "mean",  # "first" | "last" | "mean"
    use_subtoken_pooling: bool = False,
) -> str:
    device = next(model.parameters()).device

    # 1) Flair word tokenize + truncate by word budget
    words = [(t.text if hasattr(t, "text") else str(t)) for t in flair_tok.tokenize(input_text)]
    words = words[:seq_spec.max_source_length]
    if not words:
        return ""

    # 2) HF tokenize with word alignment
    enc = hf_tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=hf_tokenizer.model_max_length,
        return_attention_mask=True,
        add_special_tokens=True,
    )
    word_ids = enc.word_ids(batch_index=0)
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.inference_mode():
        if use_subtoken_pooling:
            # 3) encoder forward on subtokens
            encoder_outputs = model.get_encoder()(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                return_dict=True,
            )
            hidden = encoder_outputs.last_hidden_state[0]  # [subtok_seq_len, H]

            # 4) pool subtokens -> word-level embeddings (while preserving specials)
            pooled = _pool_subtokens_with_specials(
                hidden,
                word_ids,
                enc["attention_mask"][0],
                strategy=subtoken_pooling,
            )
            if pooled is None:
                return ""

            pooled = pooled.unsqueeze(0)  # [1, pooled_seq_len, H]
            pooled_mask = torch.ones((1, pooled.shape[1]), dtype=torch.long, device=device)

            # 5) generate from pooled encoder states
            out_ids = model.generate(
                encoder_outputs=BaseModelOutput(last_hidden_state=pooled),
                attention_mask=pooled_mask,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                no_repeat_ngram_size=3,
                repetition_penalty=1.1,
                early_stopping=True,
            )
        else:
            # Baseline generation path (no pooling) for quality sanity checks.
            out_ids = model.generate(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                no_repeat_ngram_size=3,
                repetition_penalty=1.1,
                early_stopping=True,
            )

    return hf_tokenizer.decode(out_ids[0].detach().cpu(), skip_special_tokens=True)

# ------------------------------------------------------------------
# 5) Sample predictions (baseline first)
# ------------------------------------------------------------------
print("\n--- Sample predictions (baseline first) ---")
RUN_POOLING_COMPARISON = True
sample_n = min(3, len(val_df))
for i in range(sample_n):
    inp = val_df.iloc[i]["input_text"]
    tgt = val_df.iloc[i]["target_text"]

    pred_base = generate_exit_text(
        inp,
        use_subtoken_pooling=False,
        num_beams=1,
        max_new_tokens=64,
    )

    print(f"\n[{i}] INPUT:\n{inp[:400]}...")
    print(f"TARGET:\n{tgt[:400]}...")
    print(f"PRED_BASE:\n{pred_base[:400]}...")

    if RUN_POOLING_COMPARISON:
        pred_pool = generate_exit_text(
            inp,
            use_subtoken_pooling=True,
            subtoken_pooling="mean",
            num_beams=1,
            max_new_tokens=64,
        )
        print(f"PRED_POOL:\n{pred_pool[:400]}...")


Total rows: 2197


,input_text,target_text
0,Symbol=A EntryDate=2001-02-16 Open=28.06 High=...,Symbol=A ExitDate=2001-09-27 Open=10.69 High=1...
1,Symbol=A EntryDate=2001-02-26 Open=21.15 High=...,Symbol=A ExitDate=2001-03-07 Open=23.74 High=2...
2,Symbol=A EntryDate=2001-02-26 Open=21.15 High=...,Symbol=A ExitDate=2001-03-07 Open=23.74 High=2...


Train rows: 1977
Val rows:   220


Map:   0%|          | 0/1977 [00:00<?, ? examples/s]

/Users/johnnylee/PycharmProjects/optimal_trader/modules/models/transformers/seq2seq.py:628: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
20,1.574900
40,1.095800
60,0.801700
80,0.628900
100,0.534300
120,0.521200
140,0.509100
160,0.466500
180,0.473400
200,0.452400
